# RVC Selective Weight Blending (Element-Level)

1. Computes weight differences between **reference models** (ref_a vs ref_b) to create an "emotion mask"
2. Selects TOP X% of elements with the largest differences (using RATIO)
3. Uses that mask to transfer weights from **model_a** into **model_b**
4. Linearly interpolates selected weights: `result = (1 - BLEND_FACTOR) * B + BLEND_FACTOR * A`
5. Unselected weights remain 100% model B

**Key idea:** Reference models (e.g. same speaker, different emotions) isolate emotion-related weights from voice identity. This allows transferring emotion characteristics without contaminating voice color.

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import weight_arithmetic as wa

print(f"PyTorch: {torch.__version__}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!curl -L -o /content/drive/MyDrive/speech-emotion-recognition-en.zip \
  https://www.kaggle.com/api/v1/datasets/download/dmitrybabko/speech-emotion-recognition-en

In [ ]:
# ===== CONFIGURATION =====

# Reference models: compute which weights carry the target characteristic (e.g. emotion)
# Same speaker, different emotions -> differences = pure emotion weights
# Set both to None to use MODEL_A / MODEL_B for diff computation (old behavior)
REF_A = None  # e.g., "Ravdess_F_Angry_60e_1140s.pth"
REF_B = None  # e.g., "Ravdess_F_Neutral_111e_1110s.pth"

# Blend models: source and target for actual weight transfer
MODEL_A = "Ravdess_F_Angry_60e_1140s.pth"
MODEL_B = "Anton.pth"

RATIO = 0.3
BLEND_FACTOR = 1
MODULES = None
OUTPUT_PATH = "03_1.pth"

In [ ]:
load_checkpoint = wa.load_checkpoint
extract_weights = wa.extract_weights
compute_all_differences = wa.compute_all_differences
compute_threshold = wa.compute_threshold
create_copy_masks = wa.create_copy_masks
apply_mask_blend = wa.apply_mask_blend
save_model = wa.save_model

print("Functions loaded from weight_arithmetic module.")

In [ ]:
# ===== 1. LOAD MODELS =====

cpt_a = load_checkpoint(MODEL_A)
cpt_b = load_checkpoint(MODEL_B)

# Load reference models if specified, else use blend models
if REF_A and REF_B:
    cpt_ref_a = load_checkpoint(REF_A)
    cpt_ref_b = load_checkpoint(REF_B)
    print("Using separate reference models for diff map")
else:
    cpt_ref_a = cpt_a
    cpt_ref_b = cpt_b
    print("Using blend models for diff map (no reference models set)")

# ===== 2. COMPUTE DIFFERENCES FROM REFERENCE MODELS =====

print("\nComputing weight differences at element level...")
diff_tensors, stats = compute_all_differences(cpt_ref_a, cpt_ref_b, MODULES)
print(f"Analyzed {len(diff_tensors)} layers")

# ===== 3. COMPUTE THRESHOLD AND MASKS =====

print(f"\nComputing threshold for RATIO={RATIO}...")
threshold = compute_threshold(diff_tensors, RATIO)

print("\nCreating masks for copying...")
masks, element_counts = create_copy_masks(diff_tensors, threshold)

In [ ]:
# ===== VISUALIZATION =====

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# --- Chart 1: Histogram of all differences ---
ax1 = axes[0, 0]
all_diffs = torch.cat([d.flatten() for d in diff_tensors.values()]).numpy()
ax1.hist(all_diffs, bins=100, color='steelblue', edgecolor='none', alpha=0.7, log=True)
ax1.axvline(x=threshold, color='red', linestyle='--', linewidth=2, 
            label=f'Threshold: {threshold:.6f}')
ax1.set_xlabel('Absolute Difference')
ax1.set_ylabel('Weight Count (log)')
ax1.set_title(f'Distribution of All Weight Differences (RATIO={RATIO})')
ax1.legend()

# --- Chart 2: TOP layers by number of selected elements ---
ax2 = axes[0, 1]
sorted_counts = sorted(element_counts.items(), key=lambda x: x[1], reverse=True)[:15]
layers_short = [l[:30] + "..." if len(l) > 30 else l for l, _ in sorted_counts]
counts_vals = [c for _, c in sorted_counts]
ax2.barh(range(len(layers_short)), counts_vals, color='coral')
ax2.set_yticks(range(len(layers_short)))
ax2.set_yticklabels(layers_short, fontsize=9)
ax2.set_xlabel('Number of Selected Weights') 
ax2.set_title('TOP 15 Layers with Most Selected Weights')
ax2.invert_yaxis()

# --- Chart 3: Distribution by modules ---
ax3 = axes[1, 0]
module_counts = {"enc_p": 0, "dec": 0, "flow": 0, "emb_g": 0, "other": 0}
module_totals = {"enc_p": 0, "dec": 0, "flow": 0, "emb_g": 0, "other": 0}

for layer, count in element_counts.items():
    matched = False
    for m in ["enc_p", "dec", "flow", "emb_g"]:
        if layer.startswith(m):
            module_counts[m] += count
            module_totals[m] += stats[layer]["numel"]
            matched = True
            break
    if not matched:
        module_counts["other"] += count
        module_totals["other"] += stats[layer]["numel"]

# Only modules with data
modules_with_data = [(m, c, module_totals[m]) for m, c in module_counts.items() if module_totals[m] > 0]
mod_names = [m for m, _, _ in modules_with_data]
mod_selected = [c for _, c, _ in modules_with_data]
mod_total = [t for _, _, t in modules_with_data]
mod_pct = [100 * s / t if t > 0 else 0 for s, t in zip(mod_selected, mod_total)]

x = np.arange(len(mod_names))
width = 0.35
bars1 = ax3.bar(x - width/2, mod_selected, width, label='Selected', color='coral')
bars2 = ax3.bar(x + width/2, mod_total, width, label='Total', color='steelblue', alpha=0.5)
ax3.set_ylabel('Weight Count')
ax3.set_title('Distribution of Selected Weights by Module')
ax3.set_xticks(x)
ax3.set_xticklabels(mod_names)
ax3.legend()
ax3.set_yscale('log')

# Add percentages above bars
for i, (rect, pct) in enumerate(zip(bars1, mod_pct)):
    ax3.annotate(f'{pct:.1f}%', xy=(rect.get_x() + rect.get_width()/2, rect.get_height()),
                 xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=9)

# --- Chart 4: Heatmap for layer with most selected weights ---
ax4 = axes[1, 1]
top_layer = sorted_counts[0][0] if sorted_counts else None
if top_layer and len(diff_tensors[top_layer].shape) >= 2:
    diff_2d = diff_tensors[top_layer]
    # If more than 2D, average other dimensions
    while len(diff_2d.shape) > 2:
        diff_2d = diff_2d.mean(dim=0)
    
    # Limit size for display
    max_size = 100
    if diff_2d.shape[0] > max_size:
        diff_2d = diff_2d[:max_size, :]
    if diff_2d.shape[1] > max_size:
        diff_2d = diff_2d[:, :max_size]
    
    im = ax4.imshow(diff_2d.numpy(), aspect='auto', cmap='hot')
    plt.colorbar(im, ax=ax4, label='Abs Difference')
    ax4.set_title(f'Difference Heatmap: {top_layer[:40]}...')
    ax4.set_xlabel('Dimension 1')
    ax4.set_ylabel('Dimension 0')
else:
    ax4.text(0.5, 0.5, 'N/A (1D tensor)', ha='center', va='center', transform=ax4.transAxes)
    ax4.set_title('Difference Heatmap')

plt.tight_layout()
plt.show()

# Statistics table
print(f"\nStatistics:")
print(f"  Total layers: {len(diff_tensors)}")
print(f"  Total parameters: {sum(s['numel'] for s in stats.values()):,}")
print(f"  Selected weights: {sum(element_counts.values()):,} ({100*RATIO:.2f}%)")
print(f"  Threshold: {threshold:.8f}")

In [ ]:
# ===== 4. BLEND WEIGHTS =====

print(f"Blending selected weights from model A into model B")
print(f"  Source: {MODEL_A}")
print(f"  Target: {MODEL_B}")
if REF_A and REF_B:
    print(f"  Diff map from: {REF_A} vs {REF_B}")
print(f"  Selected weights: {sum(element_counts.values()):,}")
print(f"  Blend factor: {BLEND_FACTOR} ({BLEND_FACTOR*100:.0f}% A + {(1-BLEND_FACTOR)*100:.0f}% B)")

# Blend using masks (from A into B)
result_weights = apply_mask_blend(cpt_a, cpt_b, masks, BLEND_FACTOR)

# Calculate percentage of total model
all_weights = extract_weights(cpt_b)
total_model_params = sum(w.numel() for w in all_weights.values())
blended_count = sum(element_counts.values())
pct_of_total = 100 * blended_count / total_model_params

print(f"  Weights blended: {blended_count:,} / {total_model_params:,} ({pct_of_total:.2f}% of total model)")

# Save
ref_info = ""
if REF_A and REF_B:
    ref_info = f" [diff map: {os.path.basename(REF_A)} vs {os.path.basename(REF_B)}]"
info = f"Blended {RATIO*100:.1f}% most different weights (factor={BLEND_FACTOR}) from {os.path.basename(MODEL_A)} into {os.path.basename(MODEL_B)}{ref_info}"
save_model(result_weights, cpt_b, OUTPUT_PATH, info)

print(f"\nDone!")

In [ ]:
import inference
inference.run_pipeline(
    0, 
    "Angry++.wav", # real path: /workspace/rvc-cli/input-audio.wav
    OUTPUT_PATH[:-4] + ".wav", # real path: /workspace/rvc-cli/output-audio.wav
    OUTPUT_PATH) # real path: /workspace/rvc-cli/path-to-pth-file.pth